# FPL Points Predictor — Exploration & Narrative

This notebook is the *story-telling* companion to the `src/` package, use it to walk through EDA, feature-target relationships, and model comparisons for a portfolio write-up or blog post. The actual production logic lives in `src/`; import from there rather than duplicating code here.

See `PROJECT_GUIDE.md` in the repo root for the full step-by-step build guide.

In [ ]:
import sys
sys.path.append('..')

from src.data_collection import TrainData
from src.feature_engineering import add_derived_features, drop_leaky_columns, correlated_features
from src.models import select_best_model
from src.evaluate import walk_forward_mae, permutation_importance_report
from src.optimize_team import get_predictions, select_best_xi, print_team

import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

## 1. Pull a few gameweeks and inspect

Start small (2-3 gameweeks) to confirm column names in the current season's data match what `src/data_collection.py` expects; see PROJECT_GUIDE.md §1 for why this check matters.

In [ ]:
sample = TrainData(gw=6, season='2024-25', form_range=4).df
sample.head()

## 2. Build the full training set and add derived features

In [ ]:
frames = []
for gw in range(5, 15):
    df = TrainData(gw=gw, season='2024-25', form_range=4).df.copy()
    df['gw'] = gw
    frames.append(df)
data = pd.concat(frames)
data = add_derived_features(data)
data = drop_leaky_columns(data)
data.info()

## 3. Feature-target correlations per position

This is where the per-position modelling decision gets justified visually: plot correlation heatmaps for GK/DEF/MID/FWD separately and compare which features matter for each.

In [ ]:
for position in ['GK', 'DEF', 'MID', 'FWD']:
    pos_data = data[data.position == position].drop(columns=['position', 'gw'])
    corr = pos_data.corr(numeric_only=True)['points_scored'].drop('points_scored').sort_values()
    plt.figure(figsize=(6, 8))
    sns.heatmap(corr.to_frame(), annot=True, cmap='coolwarm', fmt='.2f')
    plt.title(f'{position}: correlation with points_scored')
    plt.show()

## 4. Model selection, walk-forward validation, and interpretability

Compare random-CV MAE (optimistic) against walk-forward MAE (realistic) for each position: this comparison is one of the most interesting things to put in your README. See `PROJECT_GUIDE.md` §5c.

In [ ]:
# Example for one position — repeat for DEF/MID/FWD
gk_data = data[data.position == 'GK'].drop(columns=['position'])
gk_data_no_gw = gk_data.drop(columns=['gw'])

from sklearn.linear_model import Ridge
from sklearn.preprocessing import StandardScaler, PowerTransformer
from lightgbm import LGBMRegressor

best_model = select_best_model(gk_data_no_gw, 'GK', [Ridge(), LGBMRegressor()], [StandardScaler(), PowerTransformer(), None])

wf_mae, breakdown = walk_forward_mae(gk_data, best_model._initialize_pipeline(), gw_column='gw', min_train_gws=3)
print('Walk-forward MAE:', wf_mae)
breakdown

## 5. Predict a gameweek and build the optimal XI

Once you have a fitted `ModelClass` for each position, use `optimize_team.py` to produce the recommended squad; see `run_pipeline.py` for the scripted version of this.